# Interactive Stochastic GD (SGD) Demo

Aligned with lecture HTML behavior (deterministic data, low-iteration visibility).


### Optional dependency setup
Run this cell only if imports fail in your environment.

In [ ]:
import importlib.util

packages = [
    'numpy', 'matplotlib', 'plotly', 'ipywidgets', 'scipy', 'pandas', 'sklearn', 'seaborn'
]
missing = [p for p in packages if importlib.util.find_spec(p) is None]
if missing:
    print('Installing missing packages:', missing)
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(missing))
else:
    print('All common packages already available.')


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

METHOD = "sgd"  # batch | sgd | mini
MINI_BATCH = 8

def poly_fit(x, y, d):
    X = np.vander(x, N=d+1, increasing=True)
    A = X.T @ X + 1e-9*np.eye(d+1)
    b = X.T @ y
    return np.linalg.solve(A, b)

def pred(theta, xv):
    X = np.vander(xv, N=len(theta), increasing=True)
    return X @ theta

N = 40
x = np.linspace(-2.3, 2.3, N)
noise_base = np.array([np.sin((i+1)*2.137) + 0.3*np.cos((i+1)*0.917) for i in range(N)])

deg = widgets.IntSlider(description='degree', min=1, max=10, step=1, value=3, continuous_update=False)
noise = widgets.FloatSlider(description='noise', min=0.05, max=1.0, step=0.05, value=0.25, readout_format='.2f', continuous_update=False)
alpha = widgets.FloatSlider(description='alpha', min=0.001, max=0.3, step=0.001, value=0.05, readout_format='.3f', continuous_update=False)
max_iter = widgets.IntSlider(description='max iter', min=1, max=200, step=1, value=10, continuous_update=False)
out = widgets.Output()

rng = np.random.default_rng(7)

def grad_on_index(a, b, idx, y):
    e = (a + b*x[idx]) - y[idx]
    ga = 2.0 * np.mean(e)
    gb = 2.0 * np.mean(e * x[idx])
    return ga, gb

def render(*_):
    y = 1.2 + 0.8*x - 0.6*x*x + noise.value * noise_base
    th = poly_fit(x, y, deg.value)
    xg = np.linspace(-2.3, 2.3, 300)
    yg = pred(th, xg)

    t0 = np.linspace(-1.5, 2.5, 70)
    t1 = np.linspace(-2.5, 2.5, 70)
    T0, T1 = np.meshgrid(t0, t1)
    Z = np.zeros_like(T0)
    for i in range(T0.shape[0]):
        for j in range(T0.shape[1]):
            e = (T0[i,j] + T1[i,j]*x) - y
            Z[i,j] = np.mean(e*e)

    a, b = -1.8, 2.4
    pathx, pathy, mses = [a], [b], []
    n = len(x)

    for _ in range(max_iter.value):
        if METHOD == 'batch':
            idx = np.arange(n)
        elif METHOD == 'sgd':
            idx = np.array([rng.integers(0, n)])
        else:
            k = min(MINI_BATCH, n)
            idx = rng.choice(n, size=k, replace=False)

        ga, gb = grad_on_index(a, b, idx, y)
        a -= alpha.value * ga
        b -= alpha.value * gb

        # Track full-batch objective for fair comparison across methods.
        e_all = (a + b*x) - y
        mses.append(np.mean(e_all*e_all))
        pathx.append(a)
        pathy.append(b)

    fig = make_subplots(rows=1, cols=3, subplot_titles=(f'Linear to polynomial (degree={deg.value})', 'Contour + GD path', 'Optimization progress (full-batch MSE)'))
    fig.add_trace(go.Scatter(x=x, y=y, mode='markers', name='data'), row=1, col=1)
    fig.add_trace(go.Scatter(x=xg, y=yg, mode='lines', name='poly fit', line=dict(color='#dc2626')), row=1, col=1)

    fig.add_trace(go.Contour(x=t0, y=t1, z=Z, colorscale='Viridis', showscale=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=pathx, y=pathy, mode='lines+markers', marker=dict(size=4), line=dict(color='#ef4444'), name='GD path'), row=1, col=2)

    fig.add_trace(go.Scatter(x=np.arange(1, len(mses)+1), y=mses, mode='lines', line=dict(color='#111'), name='MSE'), row=1, col=3)

    fig.update_xaxes(title_text='x', range=[-2.5, 2.5], row=1, col=1)
    fig.update_yaxes(title_text='y', range=[-4.5, 3.5], row=1, col=1)
    fig.update_xaxes(title_text='theta0', row=1, col=2)
    fig.update_yaxes(title_text='theta1', row=1, col=2)
    fig.update_xaxes(title_text='iteration', row=1, col=3)
    fig.update_yaxes(title_text='MSE', row=1, col=3)
    fig.update_layout(height=500, showlegend=False)

    with out:
        out.clear_output(wait=True)
        fig.show()
        print(f'method={METHOD} | final theta=({a:.3f}, {b:.3f}) | alpha={alpha.value:.3f} | max_iter={max_iter.value}')

for w in [deg, noise, alpha, max_iter]:
    w.observe(render, 'value')

display(widgets.HBox([deg, noise, alpha, max_iter]))
display(out)
render()

